### Evaluierung der Datenpipeline

**Zweck:** Validierung der generierten `final.csv` auf ML-Tauglichkeit.
**Modell:** `HistGradientBoostingRegressor`
**Zielvariable:** Vorhersage der Temperatur für die Folgestunde (t+1).
**Validierungsmethode:** Chronologischer Train-Test-Split (80% Training, 20% Test) zur Verhinderung von Data Leakage bei Zeitreihen.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Daten laden
# 'zeitstempel' aus der Pipeline wird als Index gesetzt
df = pd.read_csv('final.csv', index_col='zeitstempel', parse_dates=True)

# 2. Zielvariable definieren (Temperaturvorhersage für t+1 Stunde)
# Wir verschieben die Temperatur um 1 Zeile nach oben
df['target_temp_1h'] = df['temperatur_c'].shift(-1)

# Die letzte Zeile wird ein NaN haben, da es keinen t+1 Wert gibt -> entfernen
df = df.dropna(subset=['target_temp_1h'])

# Features (X) und Target (y) trennen
X = df.drop(columns=['target_temp_1h'])
y = df['target_temp_1h']

# 3. Train-Test-Split
# Wichtig bei Zeitreihen: Keine zufällige Durchmischung, sondern chronologische Trennung.
# Erste 80% der Zeitachse für Training, letzte 20% für Testing.
split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Trainingsdaten: {len(X_train)} Zeilen")
print(f"Testdaten:      {len(X_test)} Zeilen")

# 4. Modell initialisieren und trainieren
print("\nTrainiere HistGradientBoostingRegressor...")
model = HistGradientBoostingRegressor(max_iter=100, random_state=42)
model.fit(X_train, y_train)

# 5. Vorhersage und Evaluation
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- Modellauswertung (Vorhersage Temperatur +1h) ---")
print(f"Mean Absolute Error (MAE): {mae:.3f} °C")
print(f"Root Mean Squared Error:   {rmse:.3f} °C")
print(f"R² Score:                  {r2:.4f}")

Trainingsdaten: 210356 Zeilen
Testdaten:      52589 Zeilen

Trainiere HistGradientBoostingRegressor...

--- Modellauswertung (Vorhersage Temperatur +1h) ---
Mean Absolute Error (MAE): 0.506 °C
Root Mean Squared Error:   0.755 °C
R² Score:                  0.9921


MAE (0.506 °C): Eine durchschnittliche Abweichung von rund einem halben Grad für die Prognose der Folgestunde ist ein exzellenter Wert.

RMSE (0.755 °C): Da der RMSE große Fehler stärker bestraft, zeigt die geringe Differenz zum MAE, dass das Modell kaum signifikante Ausreißer produziert. Dies belegt, dass die fehlenden Werte in der Pipeline (durch Interpolation und KNN) plausibel gefüllt wurden und keine Störsignale erzeugen.

R² (0.9921): Das Modell erklärt über 99 % der Varianz. Die extrahierten und kodierten Features (wie zyklische Windrichtung und Zeit-Features) bieten eine extrem hohe Vorhersagekraft.
